In [1]:
import pandas as pd
import plotly.express as px
import os
import sys

# Dateipfad für die Emissionsdaten (annual-co-emissions-from-oil_copy.csv)
emissions_data_path = r'C:\DataCraft\11 Datenvisualisierung\Projekt-Emission-Dashboard\Entwicklung_chris\annual-co-emissions-from-oil_copy.csv'

# Dateipfad für die Länder-Kontinent-Liste (kontinente_laender_liste.py)
continent_list_path = r'C:\DataCraft\11 Datenvisualisierung\Projekt-Emission-Dashboard\Entwicklung_chris\kontinente_laender_liste.py'


# Extrahieren des Verzeichnisses der kontinente_laender_liste.py, um sie importierbar zu machen
continent_list_dir = os.path.dirname(continent_list_path)
continent_list_filename_without_ext = os.path.splitext(os.path.basename(continent_list_path))[0]

# Fügen den Ordner zum Python-Pfad hinzu, damit wir die Datei importieren können
if continent_list_dir not in sys.path:
    sys.path.insert(0, continent_list_dir)

# --- Spaltennamen in deinen Rohdaten (bitte diese genau überprüfen!) ---
# Diese Namen MÜSSEN EXAKT mit den Überschriften in deiner CSV übereinstimmen.
# Achte auf Groß-/Kleinschreibung, Leerzeichen und Sonderzeichen wie '₂'.
YEAR_COL = 'Year' # Oder wie die Spalte mit den Jahren heißt, z.B. 'year'
ENTITY_COL = 'Entity' # Oder wie die Spalte mit den Ländern/Entitäten heißt, z.B. 'Country'
EMISSIONS_VALUE_COL = 'Annual CO₂ emissions from oil' # Oder wie die Spalte mit den Emissionswerten heißt


In [2]:
# --- 2.1 Emissionsdaten laden ---
try:
    df_emissions = pd.read_csv(emissions_data_path)
    print("✅ Emissionsdaten erfolgreich geladen.")
    print("Erste 5 Zeilen der Emissionsdaten:")
    print(df_emissions.head())
    print("\nVerfügbare Spalten in den Emissionsdaten:")
    print(df_emissions.columns.tolist())

except FileNotFoundError:
    print(f"❌ Fehler: Emissionsdatei '{emissions_data_path}' nicht gefunden.")
    print("Bitte überprüfen Sie den Pfad.")
    sys.exit() # Skript beenden
except Exception as e:
    print(f"🐛 Fehler beim Laden der Emissionsdaten: {e}")
    sys.exit()


✅ Emissionsdaten erfolgreich geladen.
Erste 5 Zeilen der Emissionsdaten:
        Entity  Year  Annual CO₂ emissions from oil
0  Afghanistan  1949                          0.000
1  Afghanistan  1950                      63203.996
2  Afghanistan  1951                      65952.000
3  Afghanistan  1952                      59892.000
4  Afghanistan  1953                      68307.000

Verfügbare Spalten in den Emissionsdaten:
['Entity', 'Year', 'Annual CO₂ emissions from oil']


In [3]:
# --- 2.2 Kontinente-Länder-Liste laden ---
country_to_continent_map = {}
try:
    # Versuche, die Python-Datei als Modul zu importieren
    continent_module = __import__(continent_list_filename_without_ext)
    if hasattr(continent_module, 'country_to_continent_map'):
        country_to_continent_map = continent_module.country_to_continent_map
        print("✅ Länder-Kontinent-Zuordnung erfolgreich geladen.")
    else:
        print(f"⚠️ Warnung: 'country_to_continent_map' nicht in '{continent_list_path}' gefunden.")
        print("Stellen Sie sicher, dass die Datei eine Variable namens 'country_to_continent_map' enthält, z.B.:")
        print("country_to_continent_map = {'Germany': 'Europe', ...}")

except ImportError:
    print(f"❌ Fehler: Kann '{continent_list_path}' nicht importieren.")
    print("Stellen Sie sicher, dass die Datei korrekt ist und keine Syntaxfehler enthält.")
    sys.exit()
except Exception as e:
    print(f"🐛 Fehler beim Laden der Länder-Kontinent-Liste: {e}")
    sys.exit()


--- DEBUG: kontinente_laender_liste.py wurde erfolgreich geladen. ---
DEBUG: 'Democratic Republic of Congo' in map? True
DEBUG: 'Faroe Islands' in map? True
DEBUG: 'Grenada' in map? True
DEBUG: Die Map hat 246 Einträge.
✅ Länder-Kontinent-Zuordnung erfolgreich geladen.


In [4]:
# --- 2.3 Kontinent-Spalte zum Emissions-DataFrame hinzufügen ---
if country_to_continent_map:
    # Verwende .map(), um die Kontinente basierend auf der Entity-Spalte hinzuzufügen
    df_emissions['Continent'] = df_emissions[ENTITY_COL].map(country_to_continent_map)

    # Prüfen, ob es Entities gibt, für die kein Kontinent gefunden wurde
    unmapped_entities = df_emissions[df_emissions['Continent'].isna()][ENTITY_COL].unique()
    if len(unmapped_entities) > 0:
        print(f"\n⚠️ Warnung: Für folgende Entitäten wurde kein Kontinent gefunden und sie werden im Plot fehlen: {unmapped_entities.tolist()}")
        print("Bitte fügen Sie diese in 'kontinente_laender_liste.py' hinzu oder korrigieren Sie die Schreibweise.")
        # Optional: Zeilen ohne Kontinent entfernen, oder behalten und als "Unbekannt" anzeigen
        # df_emissions.dropna(subset=['Continent'], inplace=True)
        # df_emissions['Continent'].fillna('Unknown', inplace=True) # Alternativ: Fülle NaN mit 'Unknown'

    print("\nDataFrame nach Hinzufügen der Kontinent-Spalte:")
    print(df_emissions.head())
    print(df_emissions['Continent'].unique())
else:
    print("\n❌ Keine Länder-Kontinent-Zuordnung verfügbar. Gruppierung nach Kontinenten nicht möglich.")
    print("Das Diagramm wird stattdessen nach 'Entity' gruppiert.")
    # Fallback, wenn keine Kontinent-Map geladen wurde
    group_by_col_for_plot = ENTITY_COL


DataFrame nach Hinzufügen der Kontinent-Spalte:
        Entity  Year  Annual CO₂ emissions from oil Continent
0  Afghanistan  1949                          0.000      Asia
1  Afghanistan  1950                      63203.996      Asia
2  Afghanistan  1951                      65952.000      Asia
3  Afghanistan  1952                      59892.000      Asia
4  Afghanistan  1953                      68307.000      Asia
['Asia' 'Africa' 'Europe' 'North America' 'Antarctica' 'South America'
 'Global Aggregates' 'Oceania']


In [7]:
# --- 3.1 Sicherstellen, dass alle benötigten Spalten vorhanden sind ---
required_cols = [YEAR_COL, ENTITY_COL, EMISSIONS_VALUE_COL, 'Continent']
missing_in_df = [col for col in required_cols if col not in df_emissions.columns]

if missing_in_df:
    print(f"\n❌ FEHLER: Folgende benötigte Spalten sind im DataFrame nicht vorhanden: {missing_in_df}")
    print("Bitte überprüfen Sie die Spaltennamen in Ihrer CSV-Datei und passen Sie YEAR_COL, ENTITY_COL, EMISSIONS_VALUE_COL am Anfang des Skripts an.")
    sys.exit()

# Wenn die Kontinent-Zuordnung erfolgreich war, gruppieren wir danach
if 'Continent' in df_emissions.columns and not df_emissions['Continent'].isnull().all():
    group_by_col_for_plot = 'Continent'
else:
    print("\n⚠️ Warnung: 'Continent'-Spalte ist leer oder nicht vorhanden, oder es wurden keine Kontinente zugeordnet.")
    print("Diagramm wird stattdessen nach 'Entity' gruppiert.")
    group_by_col_for_plot = ENTITY_COL



In [8]:

# --- 3.2 Plotly Express Diagramm erstellen ---
fig = px.line(df_emissions,
              x=YEAR_COL,
              y=EMISSIONS_VALUE_COL,
              color=group_by_col_for_plot, # Gruppiert die Linien nach Kontinent
              line_group=ENTITY_COL,      # Stellt sicher, dass jede Entität eine eigene Linie hat
              hover_name=ENTITY_COL,      # Zeigt den Entitätsnamen beim Hover an
              title=f'CO2-Emissionen aus Öl über die Zeit (Gruppiert nach {group_by_col_for_plot})',
              markers=True,
              line_shape="linear"
             )



In [13]:
# --- 3.3 Optional: Anpassungen für den Look (ähnlich Tableau) ---
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')),
                  selector=dict(mode='markers')) # Größe und Rand der Marker anpassen

fig.update_layout(
    hovermode="x unified", # Verbesserte Hover-Informationen
    xaxis_title="Jahr",
    yaxis_title="Jährliche CO₂-Emissionen aus Öl",
    legend_title_text="Gruppe",
    height=600 # Diagrammhöhe anpassen
)

# --- X-Achsenbereich anpassen ---
# Finde das minimale Jahr in den Daten, aber nur für Werte > 0
# Dies stellt sicher, dass die Achse erst beginnt, wenn sinnvolle Daten vorhanden sind.
meaningful_data_years = df_emissions[df_emissions[EMISSIONS_VALUE_COL] > 0][YEAR_COL]

if not meaningful_data_years.empty:
    min_year_data = meaningful_data_years.min()
    # Setze den Start der X-Achse 5 Jahre vor dem ersten Messwert (der > 0 ist)
    start_x_axis_range = min_year_data - 5
else:
    # Fallback, falls alle Emissionswerte 0 sind oder keine Daten vorhanden sind
    start_x_axis_range = df_emissions[YEAR_COL].min() if not df_emissions.empty else 1900 # Standardwert

# Setze den Endpunkt der X-Achse auf das maximale Jahr in den gesamten Daten
end_x_axis_range = df_emissions[YEAR_COL].max()

fig.update_xaxes(range=[start_x_axis_range, end_x_axis_range])

In [10]:
# --- Dropdown-Menü hinzufügen ---

# Hole alle einzigartigen Kontinente aus dem DataFrame
# Füge 'Alle Kontinente' als erste Option hinzu
all_continents = ['Alle Kontinente'] + sorted(df_emissions['Continent'].unique().tolist())

buttons = []

for continent in all_continents:
    if continent == 'Alle Kontinente':
        # Für "Alle Kontinente": Alle Traces sichtbar machen
        visible_traces = [True] * len(fig.data)
        title_text = f'CO2-Emissionen aus Öl über die Zeit (Gruppiert nach {group_by_col_for_plot})'
    else:
        # Für einen spezifischen Kontinent: Nur die Traces dieses Kontinents sichtbar machen
        # Plotly Express generiert Traces, deren 'legendgroup' dem Wert der 'color'-Spalte entspricht.
        visible_traces = [trace.legendgroup == continent for trace in fig.data]
        title_text = f'CO2-Emissionen aus Öl über die Zeit (Kontinent: {continent})'

    buttons.append(
        dict(
            label=continent,
            method="update",
            args=[{"visible": visible_traces}, # Sichtbarkeit der Traces anpassen
                  {"title": title_text}]       # Titel des Diagramms anpassen
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.01, # Position des Dropdowns (links oben)
            y=1.15, # Position des Dropdowns (oberhalb des Plots)
            showactive=True, # Aktuell ausgewählte Option hervorheben
            buttons=buttons,
        )
    ]
)

# --- 3.4 Diagramm anzeigen ---
print("\nDiagramm wird generiert und angezeigt... 📊")
fig.show()


Diagramm wird generiert und angezeigt... 📊
